# vLLM Support

[vLLM](https://github.com/vllm-project/vllm) is a popular library used for fast inference. By leveraging PagedAttention, dynamic batching, and Hugging Face model integration, vLLM makes inference more efficient and scalable for real-world applications.

Starting with `NNsight 0.4`, NNsight includes support for internal investigations of vLLM models.

## Setup

You will need to install `nnsight@vllmv1`, `vllm==0.12.0`, and `triton==3.5.0` to use vLLM with NNsight.

In [1]:
from IPython.display import clear_output
try:
    import google.colab
    is_colab = True
except ImportError:
    is_colab = False

if is_colab:
    %pip install -U git+https://github.com/ndif-team/nnsight@vllmv1 triton==3.5.0 vllm==0.12.0 numpy==2.2.4
clear_output()

 Next, let's load in our NNsight-supported vLLM model. You can find vLLM-supported models [here](https://docs.vllm.ai/en/stable/models/supported_models.html). For this exercise, we will use GPT-2.

 Please note that vLLM models require a GPU to run.

In [2]:
from nnsight.modeling.vllm import VLLM
from transformers import AutoTokenizer

MODEL_ID = "meta-llama/Llama-3.1-8B"

vllm = VLLM(MODEL_ID, dispatch = True) # vLLM doesn't support device mapping

# clear_output()

print(vllm)

WARNING 12-03 13:43:30 [vllm.py:1322] Current vLLM config is not set.
INFO 12-03 13:43:30 [scheduler.py:228] Chunked prefill is enabled with max_num_batched_tokens=2048.
INFO 12-03 13:43:30 [parallel_state.py:1200] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://127.0.0.1:47303 backend=gloo
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
WARNING 12-03 13:43:30 [vllm.py:1322] Current vLLM config is not set.
INFO 12-03 13:43:30 [scheduler.py:228] Chunked prefill is enabled with max_num_batched_tokens=2048.
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of conne

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


INFO 12-03 13:43:39 [default_loader.py:308] Loading weights took 2.86 seconds
INFO 12-03 13:43:40 [gpu_model_runner.py:3549] Model loading took 14.9889 GiB memory and 3.181967 seconds
INFO 12-03 13:43:42 [gpu_worker.py:359] Available KV cache memory: 26.40 GiB
INFO 12-03 13:43:43 [kv_cache_utils.py:1286] GPU KV cache size: 216,272 tokens
INFO 12-03 13:43:43 [kv_cache_utils.py:1291] Maximum concurrency for 131,072 tokens per request: 1.65x
INFO 12-03 13:43:43 [core.py:254] init engine (profile, create kv cache, warmup model) took 2.24 seconds
INFO 12-03 13:43:43 [llm.py:343] Supported tasks: ('generate',)
LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): VocabParallelEmbedding(num_embeddings=128256, embedding_dim=4096, org_vocab_size=128256, num_embeddings_padded=128256, tp_size=1)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (qkv_proj): QKVParallelLinear(in_features=4096, output_features=6144, bias=False, tp_s

## Interventions on vLLM models
We now have a vLLM model that runs with `nnsight`. Let's try applying some interventions on it.

In [3]:
neurons = [394, 5490, 8929]
prompt = "The truth is the"

mlp = vllm.model.layers[16].mlp.down_proj

with vllm.trace(prompt, remote=False):
    mlp.input = mlp.input.clone()
    mlp.input[-1, neurons] = 10 # no batch dimension
    out = vllm.output.save()
    last = out[:, -1].argmax()  # returns a tensor
    prediction = vllm.tokenizer.decode(last).save()

print(f"Prediction with vLLM: '{prediction}'")


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                    | …

Prediction with vLLM: '!'


Note that because of differences in default inference settings, and other implementation details that may be specific to your runtime stack, results may differ compared to Huggingface Transformers, even in the same intervention!

In [4]:
from nnsight import LanguageModel

neurons = [394, 5490, 8929]
prompt = "The truth is the"

lm = LanguageModel(MODEL_ID, dispatch=True, device_map="auto")
mlp = lm.model.layers[16].mlp.down_proj

with lm.trace(prompt, remote=False):
    mlp.input[:, -1, neurons] = 10                # batch dimension
    out = lm.output.save()
    last = out["logits"][:, -1].argmax()          # dict of tensors
    prediction = lm.tokenizer.decode(last).save()

print(f"Prediction with transformers: '{prediction}'")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Prediction with transformers: 'ats'


We've successfully performed an intervention on our vLLM model!

## Sampled Token Traceability
vLLM provides functionality to configure how each sequence samples its next token. Here's an example of how you can trace token sampling operations with the nnsight VLLM wrapper.

In [5]:
with vllm.trace("Madison Square Garden is located in the city of", temperature=0.8, top_p=0.95, max_tokens=3) as tracer:
    samples = list().save()
    logits = list().save()

    for ii in range(3):
        tracer.next()
        samples.append(vllm.samples.output)
        tracer.next()
        logits.append(vllm.logits.output)
    samples.save()
    logits.save()

print("Samples: ", samples)
print("Logits: ", logits) # different than samples with current sampling parameters

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                    | …

Samples:  [SamplerOutput(sampled_token_ids=tensor([[23]], device='cuda:0', dtype=torch.int32), logprobs_tensors=None), SamplerOutput(sampled_token_ids=tensor([[1115]], device='cuda:0', dtype=torch.int32), logprobs_tensors=None)]
Logits:  [tensor([[11.0625,  6.3438,  4.2812,  ..., -2.9688, -2.9688, -2.9688]],
       device='cuda:0', dtype=torch.bfloat16)]


<details>
<summary>
Note: gradients are not supported with vLLM
</summary>

vLLM speeds up inference through its paged attention mechanism. This means that accessing gradients and backward passes are not supported for vLLM models. As such, calling gradient operations when using `nnsight` vLLM wrappers will throw an error.
</details>

## Known Issues
* The vllm.LLM engine performs max_tokens + 1 forward passes which can lead to undesired behavior if you are running interventions on all iterations of multi-token generation.

In [ ]:
with vllm.trace("Hello World!", max_tokens=10) as tracer:
    outputs = list().save()

    with tracer.all():
        print("hello")
        out = vllm.output[:, -1].save() # tried with and without save
        outputs.append(out)

    outputs.save() # tried with and without this line

print(len(outputs))

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                    | …

hello
hello
hello
hello
hello
hello
hello
hello
hello
hello
hello
0
